**Projet ISD2 final:** Analyse d'une base de données Formule 1! (2000-2024)

In [47]:
# importation des bibiliothéques nécéssaires au projet 
import pandas as pd 
import numpy as np
import os

In [81]:
# chargement des fichiers de source

Data_Dir = "DataSet_F1"                                 # pointe vers le fichier source du répositoire
Output = "DataSet_F1_Final"        
NA_VALUES = ["//N", ""] # nom du fichier complet

In [82]:
def load_csv(csv_file_name):
    file_path = os.path.join(Data_Dir, csv_file_name)
    return pd.read_csv(file_path, na_values = NA_VALUES)

In [83]:
results           = load_csv("results.csv")
races             = load_csv("races.csv")
circuits          = load_csv( "circuits.csv")
drivers           = load_csv("drivers.csv")
constructors      = load_csv("constructors.csv")
status            = load_csv("status.csv")
qualifying        = load_csv("qualifying.csv")
pit_stops         = load_csv("pit_stops.csv")
driver_standings  = load_csv("driver_standings.csv")
lap_times         = load_csv("lap_times.csv")

pit_agg = pit_stops.groupby(["raceId", "driverId"]).agg(best_lap_ms = ("milliseconds", "min"), lap_std_ms = ("milliseconds", "std")).reset_index()

drivers["driver_name"] = drivers["forename"] + " " + drivers["surname"]
# mise en un seul fichier 

df = results.copy()
df = df.merge(races[["raceId", "year", "round", "circuitId", "date"]], on = "raceId", how = "left")
df = df.merge(circuits[["circuitId", "country"]], on = "circuitId", how = "left")
df = df.merge(drivers[["driverId", "dob","driver_age" ]], on = "driverId", how = "left")
df = df.merge(constructors[["constructorId", "name"]].rename(columns = {"name" : "constructor_name"}), on = "constructorId", how = "left")
df = df.merge(status.rename(columns = {"status": "status_label"}), on = "statusId", how = "left")
df = df.merge(pit_agg, on = ["raceId", "driverId"], how = "left")


#calculate driver age 
#calculate n_pit_stops


# on renomme chacune des colonnes(optionel mais pratique)

df = df.rename(columns = {
    "grid":         "grid_position",
    "positionOrder":  "finish_position",
    "points":       "points_scored",
    "laps":         "laps_completed",
    "status_label": "status",
    "country":      "circuit_country",
    "rank":         "fastest_lap_rank",
})

# choix de nos features -> 12 en total 

features = [
    "year",                                     
    "round", 
    "grid_position", 
    "finish_position",
    "points_scored",
    "laps_completed",
    "status",
    "constructor_name",
    "circuit_country",
    "driver_age",
    "pit_stop_count",
    "fastest_lap_rank",
]

df = df[features]
df = df[df["year"].between(2000, 2004)].reset_index(drop = true)        # reduit le nombre de lignes: passage de 1950-2024 à 2000-2024

df.to_csv(Output, index = False)

KeyError: "['driver_age'] not in index"